In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import joblib

# 1. Load dataset and clean column names
df = pd.read_excel('dataset_combined.xlsx')
df.columns = df.columns.str.strip()

# Clean up encoding artifacts in the 'Name' column to replace '?' with proper dashes
df['Name'] = df['Name'].str.replace(' ? ', ' - ', regex=False)
df['Name'] = df['Name'].str.replace('?', '-', regex=False)

# 2. Define features (including 'Name') and target
feature_columns = [
    'Name', 'Education Qualification', 'Gender', 'Community', 'Religion', 
    'Exservice-men', 'Disability', 'Sports', 'Annual-Percentage', 
    'Income', 'India'
]
target_column = 'Outcome'

df = df.dropna(subset=feature_columns + [target_column])

# 3. Encode all categorical columns (including Name and Outcome)
encoders = {}
for col in feature_columns + [target_column]:
    le = LabelEncoder()
    df[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

# 4. Set up X and y
encoded_features = [f'{col}_encoded' for col in feature_columns]
X = df[encoded_features]
y = df['Outcome_encoded']

# 5. Train eligibility model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training eligibility model...")
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print(f"Model Training Complete! Test Accuracy: {accuracy * 100:.2f}%")

# 6. Save model and encoders
# 6. Save model and encoders with compression
joblib.dump(model, 'scholarship_model.pkl', compress=3)
joblib.dump(encoders, 'label_encoders.pkl', compress=3)
print("Successfully saved model and updated encoders!")

Training eligibility model...
Model Training Complete! Test Accuracy: 99.98%
Successfully saved model and updated encoders!


In [2]:
import pandas as pd
import numpy as np
import joblib

model = joblib.load('scholarship_model.pkl')
encoders = joblib.load('label_encoders.pkl')

student_input = {
    'Education Qualification': 'Undergraduate',
    'Gender': 'Female',
    'Community': 'General',
    'Religion': 'Hindu',
    'Exservice-men': 'No',
    'Disability': 'No',
    'Sports': 'No',
    'Annual-Percentage': '90-100',
    'Income': 'Upto 1.5L',
    'India': 'In'
}

# Get all unique scholarship names from the encoder
all_schemes = encoders['Name'].classes_
eligible_schemes = []

for scheme in all_schemes:
    # Test this student against this specific scheme
    test_row = student_input.copy()
    test_row['Name'] = scheme
    
    encoded_vals = [
        encoders[col].transform([test_row[col]])[0] 
        for col in feature_columns
    ]
    
    X_sample = pd.DataFrame([encoded_vals], columns=[f'{col}_encoded' for col in feature_columns])
    prediction = model.predict(X_sample)[0]
    
    # Decode outcome prediction (0 or 1)
    decoded_outcome = encoders['Outcome'].inverse_transform([prediction])[0]
    
    if str(decoded_outcome) == '1':
        eligible_schemes.append(scheme)

print("Eligible Scholarship Schemes:")
for s in eligible_schemes:
    print(f"- {s}")

Eligible Scholarship Schemes:
- Glow and lovely Career Foundation Scholarship
- INSPIRE Scholarship 2022-23 - Scholarship for Higher Education (SHE)
- Pragati Scholarship - AICTE-Scholarship Scheme to Girl Child


In [3]:
import pandas as pd
import numpy as np
import joblib

# Load the trained model and encoders
model = joblib.load('scholarship_model.pkl')
encoders = joblib.load('label_encoders.pkl')

feature_columns = [
    'Name', 'Education Qualification', 'Gender', 'Community', 'Religion', 
    'Exservice-men', 'Disability', 'Sports', 'Annual-Percentage', 
    'Income', 'India'
]

# Define a test student profile
student_input = {
    'Education Qualification': 'Undergraduate',
    'Gender': 'Female',
    'Community': 'General',
    'Religion': 'Muslim',
    'Exservice-men': 'Yes',
    'Disability': 'Yes',
    'Sports': 'Yes',
    'Annual-Percentage': '80-90',
    'Income': '1.5L to 3L',
    'India': 'In'
}

# Iterate through all available scholarship schemes to find matches
all_schemes = encoders['Name'].classes_
eligible_schemes = []

for scheme in all_schemes:
    test_row = student_input.copy()
    test_row['Name'] = scheme
    
    # Encode each feature safely
    encoded_vals = []
    for col in feature_columns:
        val = str(test_row[col])
        # Handle unseen labels gracefully if any crop up
        if val in encoders[col].classes_:
            encoded_vals.append(encoders[col].transform([val])[0])
        else:
            encoded_vals.append(0) # Fallback encoding
            
    X_sample = pd.DataFrame([encoded_vals], columns=[f'{col}_encoded' for col in feature_columns])
    prediction = model.predict(X_sample)[0]
    
    # Decode the outcome (0 or 1)
    decoded_outcome = encoders['Outcome'].inverse_transform([prediction])[0]
    
    if str(decoded_outcome) == '1':
        eligible_schemes.append(scheme)

print("\nEligible Scholarship Schemes for Student Profile:")
if eligible_schemes:
    for s in eligible_schemes:
        print(f"- {s}")
else:
    print("No eligible scholarships found for this profile criteria.")


Eligible Scholarship Schemes for Student Profile:
- AAI Sports Scholarship Scheme in India 2022-23
- Glow and lovely Career Foundation Scholarship
- ONGC Sports Scholarship Scheme 2022-23


In [4]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

y_pred = model.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred, average='weighted'):.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:  99.98%
Precision: 0.9998
Recall:    0.9998
F1-Score:  0.9998

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     42610
           1       1.00      1.00      1.00      6542

    accuracy                           1.00     49152
   macro avg       1.00      1.00      1.00     49152
weighted avg       1.00      1.00      1.00     49152

Confusion Matrix:
[[42609     1]
 [    7  6535]]
